# XGBoost Shuffle Controls — GC Coordination Validation
M29 D23  |  Target: GC 214  |  Covariates: all other GCs

Three conditions per session (VR and OF1):
- **Real** — actual GC spike trains as covariates
- **Circular shift** — covariate matrix rolled by a large random offset
  (preserves autocorrelation and spatial tuning, destroys temporal coordination)
- **Bin shuffle** — covariate spike counts randomly permuted across all bins
  (destroys everything except total spike count)

**Figure layout per session:** schematic row (how each shuffle transforms the signal)
above a distribution row (pR² across shuffle repeats).

In [ ]:
import numpy as np
import pandas as pd
import pynapple as nap
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
from scipy.ndimage import gaussian_filter
from spatial_manifolds.detect_grids import *
from spatial_manifolds.mlencoding import MLencoding

import warnings; warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2
%matplotlib inline
plt.rcParams['font.family'] = 'Arial'

# ── Session to analyse — change these ────────────────────────────────────────
mouse        = 29
day          = 23
TARGET_ID    = 214
source_path  = '/Users/harryclark/Downloads/COHORT12/'
fig_path     = '/Users/harryclark/Documents/spatial-manifolds/scripts/figures/figure_validations/'

HISTORY_LENGTH = 100    # ms  — set from parameter validation
FIXED_NFILTERS = 5
N_COV          = 10     # number of GC covariate cells used in models
N_CV           = 5
N_SHUFFLES     = 100    # repeats per shuffle type
MIN_SHIFT_S    = 30.0   # minimum circular shift (seconds)
SCHEMATIC_BINS = 3000   # bins shown in schematic (~30 s at 10ms)
SCHEMATIC_START = 24150  # start bin for schematic window

COL_REAL  = '#c04744'
COL_CIRC  = '#e58e38'
COL_BIN   = '#3171ae'
COL_COV   = '#555555'

def white_to_hex_cmap(h):
    return LinearSegmentedColormap.from_list('c', ['#ffffff', h])

rng = np.random.default_rng(42)
print(f'Config ready. Target GC: {TARGET_ID}')

## 1. Load session data and GC covariate matrix

In [ ]:
print('Loading VR...')
tcs_vr, tcs_time_vr, _, last_ephys_bin_vr, beh_vr, clusters_vr = compute_vr_tcs(
    mouse, day, apply_zscore=False, apply_guassian_filter=False, source_path=source_path)
last_t_vr = clusters_vr[clusters_vr.index[0]].count(
    bin_size=time_bs, time_units='ms').index[-1]
ep_vr = nap.IntervalSet(start=0, end=last_t_vr, time_units='s')

print('Loading OF1...')
tcs_of, tcs_time_of, beh_of, clusters_of, ep_of = compute_of_tcs(
    mouse, day, apply_zscore=False, apply_guassian_filter=False,
    source_path=source_path, session='OF1')

gcs, ngs, all_cells = classify_cells_both_sessions(mouse, day, source_path=source_path)
all_gc_ids = [int(c) for c in gcs.cluster_id.values
              if c != TARGET_ID and c in tcs_time_vr]
# Use N_COV randomly sampled GC cells as covariates
gc_ids = list(rng.choice(all_gc_ids,
                         size=min(N_COV, len(all_gc_ids)),
                         replace=False).astype(int))
print(f'GC covariate cells (selected {len(gc_ids)} of {len(all_gc_ids)} available)')

def _pad(arr, T):
    arr = np.array(arr)[:T]; return np.pad(arr, (0, max(0, T-len(arr))))

# ── VR signals ───────────────────────────────────────────────────────────────
y_vr  = np.array(tcs_time_vr[TARGET_ID]);  T_vr = len(y_vr)
dt    = np.array(beh_vr['travel'].bin_average(bin_size=time_bs, time_units='ms', ep=ep_vr)
                 - ((beh_vr['trial_number'][0]-1)*tl))
dt    = pd.Series(dt).ffill().bfill().values
pos_vr = dt % tl

# GC covariate matrix (VR) — no position
gc_mat_vr = np.vstack([_pad(np.array(tcs_time_vr[c]), T_vr) for c in gc_ids]).T

# ── OF1 signals ───────────────────────────────────────────────────────────────
y_of  = np.array(tcs_time_of[TARGET_ID]);  T_of = len(y_of)
gc_ids_of = [c for c in gc_ids if c in tcs_time_of]  # same N_COV cells
gc_mat_of = np.vstack([_pad(np.array(tcs_time_of[c]), T_of) for c in gc_ids_of]).T

print(f'VR:  T={T_vr} bins, gc_mat={gc_mat_vr.shape}')
print(f'OF1: T={T_of} bins, gc_mat={gc_mat_of.shape}')

## 2. Shuffle helpers

In [ ]:
MIN_SHIFT_BINS = int(MIN_SHIFT_S * 1000 / time_bs)

def circular_shift(mat, rng):
    """Roll all covariate columns by the same random offset."""
    T = mat.shape[0]
    shift = rng.integers(MIN_SHIFT_BINS, T // 2)
    if rng.random() < 0.5: shift = -shift
    return np.roll(mat, shift, axis=0)

def bin_shuffle(mat, rng):
    """Randomly permute all time bins for each covariate independently."""
    out = mat.copy()
    for j in range(out.shape[1]):
        out[:, j] = rng.permutation(out[:, j])
    return out

print('Shuffle helpers defined.')

## 3. Compute pR² — real, circular shift, bin shuffle

In [ ]:
xgb = MLencoding(tunemodel='xgboost', cov_history=True, spike_history=False,
                  window=time_bs, n_filters=FIXED_NFILTERS, max_time=HISTORY_LENGTH)

results = {}   # results[session] = {'real': float, 'circ': array, 'bin': array}

for sess, y, gc_mat in [('VR', y_vr, gc_mat_vr), ('OF1', y_of, gc_mat_of)]:
    print(f'\n── {sess} ──')

    # Real
    _, pr2_real = xgb.fit_cv(gc_mat, y, verbose=0, continuous_folds=True, n_cv=N_CV)
    real = float(np.nanmean(pr2_real))
    print(f'  Real pR²={real:+.4f}')

    # Circular shift shuffles
    circ_pr2s = []
    for si in range(N_SHUFFLES):
        mat_s = circular_shift(gc_mat, rng)
        _, pr2 = xgb.fit_cv(mat_s, y, verbose=0, continuous_folds=True, n_cv=N_CV)
        circ_pr2s.append(float(np.nanmean(pr2)))
        if (si+1) % 25 == 0:
            print(f'  Circ  {si+1}/{N_SHUFFLES}  mean so far={np.mean(circ_pr2s):+.4f}', end='\r')
    print(f'  Circular shift  mean={np.mean(circ_pr2s):+.4f}  '
          f'95th={np.nanpercentile(circ_pr2s,95):+.4f}')

    # Bin shuffle
    bin_pr2s = []
    for si in range(N_SHUFFLES):
        mat_s = bin_shuffle(gc_mat, rng)
        _, pr2 = xgb.fit_cv(mat_s, y, verbose=0, continuous_folds=True, n_cv=N_CV)
        bin_pr2s.append(float(np.nanmean(pr2)))
        if (si+1) % 25 == 0:
            print(f'  Bin   {si+1}/{N_SHUFFLES}  mean so far={np.mean(bin_pr2s):+.4f}', end='\r')
    print(f'  Bin shuffle     mean={np.mean(bin_pr2s):+.4f}  '
          f'95th={np.nanpercentile(bin_pr2s,95):+.4f}')

    results[sess] = {
        'real': real,
        'circ': np.array(circ_pr2s),
        'bin':  np.array(bin_pr2s),
    }

print('\nDone.')

## 4. Figure — schematic + distributions

In [ ]:
# ── Schematic helper ─────────────────────────────────────────────────────────
N_COV_SHOW = 10   # number of covariate traces shown in schematic

def draw_schematic(ax_target, ax_real, ax_circ, ax_bin,
                    y, gc_mat, window_start, window_len, rng_s):
    """Show target + 10 covariate cells in three shuffle conditions.
    Each covariate is a separate row within the axis, stacked vertically.
    """
    ws, we = window_start, window_start + window_len
    t  = np.arange(ws, we) * time_bs / 1000   # seconds

    # Pick N_COV_SHOW covariate cells with most spikes in the window
    n_spikes_window = (gc_mat[ws:we] > 0).sum(axis=0)
    top_idx = np.argsort(n_spikes_window)[::-1][:N_COV_SHOW]

    target_seg  = y[ws:we]
    mat_real    = gc_mat[ws:we][:, top_idx]          # (window, N_COV_SHOW)
    mat_circ    = circular_shift(gc_mat, rng_s)[ws:we][:, top_idx]
    mat_bin     = bin_shuffle(gc_mat, rng_s)[ws:we][:, top_idx]

    def raster_multi(ax, mat, color, label, show_xlabel=False):
        """Stack N_COV_SHOW raster rows within one axis."""
        n = mat.shape[1]
        for row in range(n):
            spike_t = t[mat[:, row] > 0]
            ax.vlines(spike_t, row, row + 0.8, color=color, lw=0.7, alpha=0.75)
        ax.set_ylim(-0.2, n)
        ax.set_yticks([])
        ax.set_ylabel(label, fontsize=7, rotation=0, labelpad=45, va='center')
        ax.spines[:].set_visible(False)
        ax.tick_params(bottom=False, labelbottom=show_xlabel)
        if show_xlabel:
            ax.set_xlabel('Time (s)', fontsize=7)
            ax.tick_params(labelbottom=True, labelsize=6)
        ax.set_xlim(t[0], t[-1])

    def raster_single(ax, sig, color, label):
        spike_t = t[sig > 0]
        ax.vlines(spike_t, 0, 0.8, color=color, lw=0.8, alpha=0.9)
        ax.set_ylim(-0.1, 1.0)
        ax.set_yticks([])
        ax.set_ylabel(label, fontsize=7, rotation=0, labelpad=45, va='center')
        ax.spines[:].set_visible(False)
        ax.tick_params(bottom=False, labelbottom=False)
        ax.set_xlim(t[0], t[-1])

    raster_single(ax_target, target_seg, 'black', f'Target\nGC {TARGET_ID}')
    raster_multi(ax_real, mat_real, COL_REAL, f'Real\n({N_COV_SHOW} GCs)')
    raster_multi(ax_circ, mat_circ, COL_CIRC, f'Circular\nshift')
    raster_multi(ax_bin,  mat_bin,  COL_BIN,  f'Bin\nshuffle', show_xlabel=True)

    # Dashed dividers between panels
    for ax in [ax_real, ax_circ]:
        ax.axhline(0, color='#dddddd', lw=0.6)

    # 1-s scale bar on target
    bar_bins = int(1000 / time_bs)
    ax_bin.plot([t[-1] - 1, t[-1]], [-0.15, -0.15], 'k-', lw=2,
                clip_on=False, transform=ax_bin.get_xaxis_transform())
    ax_bin.text(t[-1] - 0.5, -0.30, '1 s', ha='center', va='top',
                fontsize=6, transform=ax_bin.get_xaxis_transform())


# ── Build full figure ─────────────────────────────────────────────────────────
SESSIONS = [('VR', y_vr, gc_mat_vr), ('OF1', y_of, gc_mat_of)]

fig = plt.figure(figsize=(14, 10))
gs_outer = gridspec.GridSpec(2, 2, figure=fig,
                              hspace=0.45, wspace=0.35,
                              left=0.12, right=0.97,
                              top=0.93, bottom=0.07)

for col, (sess, y, gc_mat) in enumerate(SESSIONS):
    res  = results[sess]
    rng_s = np.random.default_rng(7)   # fixed seed for schematic shuffles

    # ── Top half: schematic (4 raster rows) ──────────────────────────────────
    gs_sch = gridspec.GridSpecFromSubplotSpec(
        4, 1, subplot_spec=gs_outer[0, col],
        hspace=0.08, height_ratios=[1, 3, 3, 3])
    ax_tgt  = fig.add_subplot(gs_sch[0])
    ax_real = fig.add_subplot(gs_sch[1])
    ax_circ = fig.add_subplot(gs_sch[2])
    ax_bin  = fig.add_subplot(gs_sch[3])

    draw_schematic(ax_tgt, ax_real, ax_circ, ax_bin,
                   y, gc_mat, SCHEMATIC_START, SCHEMATIC_BINS, rng_s)

    ax_tgt.set_title(f'{sess} — spike rasters\n(one representative covariate GC)',
                     fontsize=8.5, fontweight='bold', pad=6)

    # Condition labels with colour patches
    for ax, label, col_c in [
        (ax_real, 'Real: temporal coordination intact', COL_REAL),
        (ax_circ, 'Circular shift: coordination destroyed,\nspatial tuning preserved', COL_CIRC),
        (ax_bin,  'Bin shuffle: all structure destroyed,\nspike count preserved', COL_BIN),
    ]:
        ax.text(1.01, 0.5, label, transform=ax.transAxes,
                fontsize=5.5, va='center', color=col_c, style='italic')

    # ── Bottom half: pR² distribution ────────────────────────────────────────
    ax_dist = fig.add_subplot(gs_outer[1, col])

    bins = np.linspace(
        min(res['bin'].min(), res['circ'].min()) - 0.005,
        max(res['bin'].max(), res['circ'].max(), res['real']) + 0.005,
        35)

    ax_dist.hist(res['bin'],  bins=bins, color=COL_BIN,  alpha=0.65, label='Bin shuffle',
                 edgecolor='white', linewidth=0.3)
    ax_dist.hist(res['circ'], bins=bins, color=COL_CIRC, alpha=0.65, label='Circular shift',
                 edgecolor='white', linewidth=0.3)
    ax_dist.axvline(res['real'], color=COL_REAL, lw=2.5, label=f'Real  pR²={res["real"]:+.4f}',
                    zorder=5)

    # 95th percentile lines
    ax_dist.axvline(np.nanpercentile(res['circ'], 95), color=COL_CIRC, lw=1.2,
                    ls='--', alpha=0.8, label=f'Circ 95th pct')
    ax_dist.axvline(np.nanpercentile(res['bin'],  95), color=COL_BIN,  lw=1.2,
                    ls='--', alpha=0.8, label=f'Bin 95th pct')

    ax_dist.axvline(0, color='#aaaaaa', lw=0.8, ls=':')
    ax_dist.set_xlabel('pR²', fontsize=9)
    ax_dist.set_ylabel(f'Count  (n={N_SHUFFLES} shuffles)', fontsize=9)
    ax_dist.set_title(
        f'{sess} — pR² distribution\n'
        f'Real={res["real"]:+.4f}  '
        f'Circ mean={res["circ"].mean():+.4f}  '
        f'Bin mean={res["bin"].mean():+.4f}',
        fontsize=8.5, fontweight='bold')
    ax_dist.legend(fontsize=7, frameon=False)
    ax_dist.spines[['top','right']].set_visible(False)
    ax_dist.tick_params(labelsize=8)

fig.suptitle(
    f'M{mouse} D{day}  GC {TARGET_ID} — shuffle control validation\n'
    f'Covariates: all other GCs (n={len(gc_ids)})  '
    f'|  history={HISTORY_LENGTH}ms  n_filters={FIXED_NFILTERS}  '
    f'|  {N_SHUFFLES} shuffles each',
    fontsize=10, fontweight='bold'
)

save_path = fig_path + f'shuffle_controls_M{mouse}D{day}_GC{TARGET_ID}.pdf'
fig.savefig(save_path, bbox_inches='tight', dpi=300)
plt.show()
print(f'Saved → {save_path}')